# Projeto Final — Aprendizado de Máquina
## Predição de Depressão em Estudantes

**Disciplina:** Aprendizado de Máquina — DCOMP/UFS


## 5.1 Identificação e descrição do problema

**Título:** Predição de Depressão em Estudantes

**Integrantes:**
- @EvilynAquino
- @Mat-Macedo
- @luanorama

**Fonte dos dados:** [Student Depression Dataset — Kaggle](https://www.kaggle.com/datasets/hopesb/student-depression-dataset)

**Objetivo:** Prever se um estudante apresenta indícios de depressão a partir de características demográficas, acadêmicas e de estilo de vida, contribuindo para a identificação precoce de casos de risco.

**Atributo-alvo:** `Depression` (0 = não, 1 = sim)

**Atributos preditivos:** idade, gênero, cidade, profissão, pressão acadêmica/trabalho, CGPA, satisfação com estudo/trabalho, duração do sono, hábitos alimentares, grau acadêmico, horas de estudo/trabalho, estresse financeiro, histórico familiar de doença mental, pensamentos suicidas prévios.

**Tipo da tarefa:** Classificação binária (o atributo-alvo é categórico: 0 ou 1).

> ⚠️ **Nota ética:** este dataset trata de saúde mental. A coluna sobre pensamentos suicidas é um dado sensível — vamos discutir isso criticamente na seção de discussão final, e não apenas usá-la como mais uma feature.


In [ ]:
# Bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")


In [ ]:
# Carregar os dados diretamente do repositório GitHub (sem depender de arquivo local)
# IMPORTANTE: depois de subir o repo, troque a URL abaixo pela URL "raw" real do seu GitHub
URL = "https://raw.githubusercontent.com/EvilynAquino/student-depression-prediction/refs/heads/main/data/student_depression_dataset.csv"

df = pd.read_csv(URL)
df.head()


## 5.2 Compreensão dos dados

Nesta seção, vamos analisar e **interpretar** (não só executar comandos):
- Quantidade de registros e atributos
- Tipos das variáveis
- Valores ausentes
- Duplicações
- Inconsistências
- Distribuição do atributo-alvo
- Desbalanceamento


In [ ]:
print(f"Registros: {df.shape[0]}")
print(f"Atributos: {df.shape[1]}")
df.info()


In [ ]:
# Valores ausentes
df.isnull().sum().sort_values(ascending=False)


In [ ]:
# Duplicações
print(f"Linhas duplicadas: {df.duplicated().sum()}")


In [ ]:
# Distribuição do atributo-alvo
df['Depression'].value_counts(normalize=True) * 100


**Interpretação:** *(escrever aqui a interpretação dos resultados acima — não deixar em branco, isso é cobrado na avaliação)*


## 5.3 Análise exploratória

Toda figura/tabela abaixo deve vir acompanhada de uma explicação escrita logo depois.


In [ ]:
# Exemplo: distribuição de idade por classe
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(data=df, x='Age', hue='Depression', kde=True, ax=ax)
ax.set_title('Distribuição de idade por classe de depressão')
plt.show()


**Interpretação:** *(explicar o que o gráfico mostra)*


In [ ]:
# Exemplo: correlação entre variáveis numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.title('Matriz de correlação')
plt.show()


**Interpretação:** *(explicar quais variáveis mais se relacionam com o alvo e por quê)*

*(Adicionar mais análises: boxplots, gráficos de dispersão, tabelas de frequência para variáveis categóricas, etc — sempre com interpretação)*


## 5.4 Pré-processamento

Para cada tratamento aplicado, explicar: (1) qual problema foi encontrado, (2) qual tratamento foi aplicado, (3) por que esse tratamento foi escolhido.

> ⚠️ Cuidado para não vazar informação entre treino e teste (ajustar `scaler`/`encoder` só com dados de treino).


In [ ]:
# Remover colunas não preditivas
df_proc = df.drop(columns=['id'])

In [ ]:
# Tratamento de valores ausentes
# Sendo 'Financial Stress' uma coluna com pouquíssimos ausentes (cerca de 1-2 registros),
# a abordagem mais segura e simples é descartar as linhas nulas, não interferindo
# na distribuição natural dos dados por meio de imputação.
df_proc = df_proc.dropna(subset=['Financial Stress'])

In [ ]:
# Codificação de variáveis categóricas
# Atenção: 'Sleep Duration' tem ordem natural — considerar encoding ordinal em vez de one-hot puro

# 1. Ordinal Encoding para colunas ordenadas
sleep_map = {'Less than 5 hours': 1, '5-6 hours': 2, '7-8 hours': 3, 'More than 8 hours': 4, 'Others': 0}
diet_map = {'Unhealthy': 1, 'Moderate': 2, 'Healthy': 3, 'Others': 0}

if 'Sleep Duration' in df_proc.columns:
    df_proc['Sleep Duration'] = df_proc['Sleep Duration'].map(sleep_map).fillna(0)

if 'Dietary Habits' in df_proc.columns:
    df_proc['Dietary Habits'] = df_proc['Dietary Habits'].map(diet_map).fillna(0)

# 2. Mapeamento binário para Yes/No
binary_cols = ['Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']
for col in binary_cols:
    if col in df_proc.columns:
        df_proc[col] = df_proc[col].map({'Yes': 1, 'No': 0}).fillna(0)

# 3. One-Hot Encoding para as demais colunas categóricas (Gender, City, Profession, etc)
cat_cols = df_proc.select_dtypes(include=['object']).columns
df_proc = pd.get_dummies(df_proc, columns=cat_cols, drop_first=True)

**Justificativa dos tratamentos:** 
- **Remoção do `id`**: O `id` não possui capacidade preditiva e pode fazer o modelo memorizar registros, além de atrapalhar na generalização.
- **Tratamento de nulos em `Financial Stress`**: Foi usada a remoção das linhas nulas (`.dropna()`) porque os valores ausentes são irrisórios na base e assim preservamos os dados reais sem inventar (imputar) um valor que poderia gerar viés.
- **Encoding Ordinal em `Sleep Duration` e `Dietary Habits`**: Aplicado porque são variáveis cuja natureza contém uma gradação de valor/qualidade, sendo útil preservar esta ordem numérica para algoritmos baseados em árvore ou lineares.
- **Mapeamento Binário**: Variáveis de tipo Sim/Não puderam ser convertidas diretamente para 1 e 0.
- **One-Hot Encoding (com `drop_first=True`)**: Usado nas demais colunas (ex: Gênero, Profissão, Cidade), convertendo categorias não-ordenadas em binárias. Usamos `drop_first` para evitar multicolinearidade (efeito "dummy trap").


## 5.5 Separação dos dados

- Separar treino e teste
- Justificar a proporção utilizada
- Utilizar estratificação (o alvo é levemente desbalanceado: ~59%/41%)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

X = df_proc.drop(columns=['Depression'])
y = df_proc['Depression']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Escalonamento (Scaling)
scaler = StandardScaler()
# Ajustamos o scaler APENAS no X_train para evitar data leakage
X_train_scaled = scaler.fit_transform(X_train)
# Transformamos o X_test com os mesmos parâmetros
X_test_scaled = scaler.transform(X_test)

# Voltando para DataFrame para manter legibilidade das colunas (opcional, mas recomendado)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)


**Justificativa:** 
A proporção escolhida (80% treino / 20% teste) é um padrão da literatura que maximiza o conjunto de aprendizado (80%), provendo exemplos suficientes para o treinamento do modelo, enquanto ainda reserva 20% para uma validação sólida e confiável.

A adição do parâmetro `stratify=y` é imprescindível aqui. Como constatado previamente, as classes têm certo desbalanceamento (~59% da classe majoritária). Se a separação fosse puramente aleatória, correríamos o risco de gerar um conjunto de teste com proporções de classes anômalas, o que distorceria a precisão e o F1-Score do modelo. O `stratify=y` preserva esta mesma proporção de ~59/41 nos subconjuntos criados.

**Escalonamento (StandardScaler):** Modelos lineares como o `SGDClassifier` são altamente sensíveis à escala das variáveis. Para evitar que variáveis com valores absolutos maiores (como Idade ou CGPA) dominem variáveis menores, usamos o `StandardScaler` (média 0 e variância 1). É **crucial** que o `.fit()` tenha sido aplicado *somente* nos dados de treino (`X_train`), prevenindo qualquer vazamento de informação (*data leakage*) dos dados de teste para o modelo.

## 5.6 Modelagem

Modelos mínimos exigidos para classificação:
- Baseline (DummyClassifier)
- SGDClassifier
- RandomForestClassifier


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

baseline = DummyClassifier(strategy='most_frequent', random_state=42)
sgd = SGDClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

modelos = {'Baseline': baseline, 'SGDClassifier': sgd, 'RandomForest': rf}

for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring='f1')
    print(f"{nome}: F1 médio = {scores.mean():.3f} (+/- {scores.std():.3f})")


**Comparação e escolha do modelo final:** *(justificar qual modelo performou melhor e por quê)*


## 5.7 Avaliação e discussão

Métricas para classificação: matriz de confusão, acurácia, precisão, revocação, F1-score.


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

modelo_final = rf  # ajustar conforme escolha da seção anterior
modelo_final.fit(X_train, y_train)
y_pred = modelo_final.predict(X_test)

print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()


**Discussão:**
- Qual modelo apresentou o melhor resultado e por quê: *(escrever)*
- Quais erros foram observados: *(escrever)*
- Quais limitações existem: *(escrever)*
- O que poderia ser melhorado: *(escrever)*
- Reflexão ética sobre o uso da variável de pensamentos suicidas como preditor: *(escrever)*
